# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hidatara-ds/flyrank-intern-ml-gilang/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

We examine two core findings published in the FlyRank research report (*The State of AI-Driven SEO, March 2026*, `docs/flyrank-seo-research-march-2026.pdf`) through the lens of leakage taxonomy and validation rigor:

---

### Finding 1: The Freshness Multiplier & Age-Freshness Matrix (Paper Finding #4 & #8)
- **Paper Finding:** The paper concludes that the 31–90 day window is the strongest stable freshness band with an observed growth-to-decline ratio of **7.88 : 1**. In the extreme tail (`361+` days untouched), the report charts a growth ratio of **283 : 1**, noting that while the sample is small, only 1 declining page existed in that bucket.
- **Where does the label come from?** The label is retrospective growth-to-decline derived from `trend_direction` (and underlying `trend_pct`), which calculates the percentage change in impressions between the final 30 days and the preceding 30 days within the same trailing 90-day observation window.
- **Methodology & Validation Questions (Constructive Audit):**
  1. *Survivorship / Selection Filter Effect:* The paper notes in its methodology that ML feature vectors were sampled from active content with `sessions > 0 and impressions > 0`. In the 361+ day tier, legacy articles that completely died (dropped to 0 impressions) are silently excluded from this active cut. The dramatic 283:1 ratio in the extreme tail reflects severe survivorship selection—only the single most resilient evergreen article remained in the sample. How does the freshness ratio look when computed over the entire portfolio without conditioning on active survival?
  2. *Domain / Client Confounding:* The reported ratios aggregate all 57 client brands into a single pooled distribution without grouping by `client_id`. If high-authority enterprise domains happen to publish content heavily in the 31–90 day band, does the 7.88:1 growth ratio reflect the freshness window itself, or domain-level publishing cadence?
  3. *Directional Correlation vs. Forward Decision-Support:* Does being in the 31–90 day freshness tier causally predict that an article will *continue* growing over the next 30 days ($T \to T+30\text{d}$), or does it merely capture the natural indexing trajectory of newly launched pages?

---

### Finding 2: Growth Classification via Logistic Regression (Paper ML Appendix)
- **Paper Finding:** The paper presents a Logistic Regression model achieving **71% holdout accuracy** in classifying growing from declining pages, highlighting content age as the primary negative signal and visibility/impressions as top positive signals.
- **Where does the label come from?** The target is a binary classification of `trend_direction` (`growing` vs. `declining`), derived directly from `trend_pct` over the trailing 30-day window.
- **Methodology & Validation Questions (Constructive Audit):**
  1. *Unstratified Random Split vs. Client Leakage:* The methodology discloses an unstratified *80/20 train/test split* across 61.8k pieces. A random row split allows URLs from the same website domain to populate both training and test partitions. The model can easily memorize site-level baseline authority and traffic scale, producing an optimistic holdout accuracy that degrades when applied to a brand-new client.
  2. *Missing Baseline Class Prevalence (Base Rate):* The paper reports a headline accuracy of 71% without reporting the baseline class distribution of growing pages in that active sample. In search datasets filtered for active engagement, if 60%–65% of pages experienced positive momentum, 71% accuracy represents only 6 to 11 percentage points of true discriminative lift over a naive majority class predictor.
  3. *Overlapping Temporal Windows:* Features such as `days_with_impressions` and trailing 90-day volume overlap the 30-day window during which growth was measured. A page that experienced positive growth during the outcome window inherently accumulated higher active days within that exact window, introducing subtle target-window leakage.


In [1]:
# Section 1 Code: Empirical Check on Base Rates, Active Subsets, and Freshness Tiers
import os
import pandas as pd
import numpy as np

# Navigate to repo root
while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

data_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(data_path), f"Dataset not found at {data_path}"
df = pd.read_csv(data_path)

print("=== Audit 1: Class Distribution & Base Rates ===")
total_rows = len(df)
active_mask = (df["impressions_90d"] > 0) & (df["sessions_90d"] > 0)
df_active = df[active_mask]

print(f"Full Dataset Size:   {total_rows:,} rows across {df['client_id'].nunique()} clients")
print(f"Active Content Cut:  {len(df_active):,} rows ({len(df_active)/total_rows:.1%} of portfolio)")

print("\nClass Distributions (trend_direction):")
full_dist = df["trend_direction"].value_counts(normalize=True).rename("Full Portfolio")
active_dist = df_active["trend_direction"].value_counts(normalize=True).rename("Active Sample")
dist_table = pd.concat([full_dist, active_dist], axis=1).map(lambda x: f"{x:.2%}")
print(dist_table)

print("\n=== Audit 2: Freshness Tier Ratios & Tail Vulnerability ===")
ct_counts = pd.crosstab(df["freshness_tier"], df["trend_direction"])
ct_counts["up_to_down_ratio"] = (ct_counts["up"] / ct_counts["down"]).round(2)
ct_counts["n_total"] = ct_counts[["up", "down", "flat"]].sum(axis=1)
print(ct_counts[["n_total", "up", "down", "flat", "up_to_down_ratio"]])


=== Audit 1: Class Distribution & Base Rates ===
Full Dataset Size:   30,000 rows across 32 clients
Active Content Cut:  30,000 rows (100.0% of portfolio)

Class Distributions (trend_direction):
                Full Portfolio Active Sample
trend_direction                             
down                    54.21%        54.21%
stable                  19.87%        19.87%
up                      14.63%        14.63%
new                      7.45%         7.45%
flat                     3.84%         3.84%

=== Audit 2: Freshness Tier Ratios & Tail Vulnerability ===
trend_direction  n_total    up   down  flat  up_to_down_ratio
freshness_tier                                               
0-30               14539  3168  10473   898              0.30
181+                 125    27     82    16              0.33
31-90                142    38    103     1              0.37
91-180              6996  1155   5604   237              0.21


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Why Split Architecture Dictates Model Truthfulness

In search analytics, data points are **hierarchically grouped by client domains**. Pages belonging to the same client share domain authority, backlink velocity, CMS infrastructure, and topical niches.

1. **The Naive Random Split ("Before"):**
   - Standard row-wise random splitting (`train_test_split(test_size=0.2, random_state=42)`) scatters pages from the same client across both train and test partitions.
   - The model learns client-specific signatures (e.g., specific search volume tiers or CTR curves unique to Client A) and effectively memorizes site-level outcomes (**cross-client data leakage**).
   - This produces an artificially inflated operational score.

2. **The Honest Client-Holdout Split ("After"):**
   - Following our Week-5 design, we implement a **Grouped Split by `client_id`**.
   - Exactly **20% of clients** (6 clients, 2,325 content items) are completely withheld from training.
   - The model is evaluated strictly on **unseen client domains**, answering the authentic deployment question: *"Can this model rank decaying content on a client website it has never seen before?"*

### The Memorization Gap (Before vs. After)
The difference between the Naive Random Split score and the Honest Client-Holdout score is not a defect—it is the **empirical measurement of domain memorization**.


In [2]:
# Section 2 Code: Re-running Week-5 Model under Naive Random Split vs Honest Client-Holdout Split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Prepare strictly observable prior-window features (quarantining target and leaky columns)
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "age_tier_order", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier"
]

# Derive ground-truth proxy target
y = (df["trend_direction"].str.lower() == "down").astype(int).values

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    X_num[f"log_{col}"] = np.log1p(X_num[col])

X_cat = pd.get_dummies(df[categorical_features].fillna("unknown").astype(str), drop_first=True)
X_honest = pd.concat([X_num, X_cat], axis=1)

# Helper function to compute ranking metrics
def evaluate_ranking(y_true, y_probs):
    top10_idx = np.argsort(y_probs)[::-1][:10]
    top20_idx = np.argsort(y_probs)[::-1][:20]
    top50_idx = np.argsort(y_probs)[::-1][:50]
    return {
        "Base Rate": round(float(np.mean(y_true)), 3),
        "ROC-AUC": round(float(roc_auc_score(y_true, y_probs)), 3),
        "PR-AUC": round(float(average_precision_score(y_true, y_probs)), 3),
        "Precision@10": round(float(np.mean(y_true[top10_idx])), 3),
        "Precision@20": round(float(np.mean(y_true[top20_idx])), 3),
        "Precision@50": round(float(np.mean(y_true[top50_idx])), 3)
    }

# --- EVALUATION 1: Naive Random Row Split (Before) ---
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd = train_test_split(
    X_honest, y, test_size=0.2, random_state=42
)
rf_naive = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=25, random_state=42, n_jobs=-1
)
rf_naive.fit(X_tr_rnd, y_tr_rnd)
probs_naive = rf_naive.predict_proba(X_te_rnd)[:, 1]
res_naive = evaluate_ranking(y_te_rnd, probs_naive)

# --- EVALUATION 2: Honest Client-Holdout Group Split (After) ---
client_series = df["client_id"].astype(str)
unique_clients = client_series.unique()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
train_clients = set(shuffled_clients[test_client_count:])

test_mask = client_series.isin(test_clients).values
train_mask = ~test_mask

rf_honest = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=25, random_state=42, n_jobs=-1
)
rf_honest.fit(X_honest[train_mask], y[train_mask])
probs_honest = rf_honest.predict_proba(X_honest[test_mask])[:, 1]
res_honest = evaluate_ranking(y[test_mask], probs_honest)

# --- COMPARISON TABLE ---
comparison_df = pd.DataFrame([res_naive, res_honest], index=[
    "Naive Random Split (Before)",
    "Honest Client-Holdout Split (After)"
])
gap_row = {
    "Base Rate": round(res_naive["Base Rate"] - res_honest["Base Rate"], 3),
    "ROC-AUC": round(res_naive["ROC-AUC"] - res_honest["ROC-AUC"], 3),
    "PR-AUC": round(res_naive["PR-AUC"] - res_honest["PR-AUC"], 3),
    "Precision@10": round(res_naive["Precision@10"] - res_honest["Precision@10"], 3),
    "Precision@20": round(res_naive["Precision@20"] - res_honest["Precision@20"], 3),
    "Precision@50": round(res_naive["Precision@50"] - res_honest["Precision@50"], 3),
}
comparison_df.loc["Memorization Gap (Before - After)"] = gap_row

print("=== Split Audit: Before vs After Performance ===")
print(comparison_df.to_string())
print(f"\nTest Set Sizes: Naive Random = {len(y_te_rnd):,} rows | Client-Holdout = {len(y[test_mask]):,} rows ({len(test_clients)} unseen clients)")


=== Split Audit: Before vs After Performance ===
                                     Base Rate  ROC-AUC  PR-AUC  Precision@10  Precision@20  Precision@50
Naive Random Split (Before)              0.545    0.767   0.784           1.0          0.95          0.94
Honest Client-Holdout Split (After)      0.391    0.754   0.626           0.7          0.75          0.76
Memorization Gap (Before - After)        0.154    0.013   0.158           0.3          0.20          0.18

Test Set Sizes: Naive Random = 6,000 rows | Client-Holdout = 2,325 rows (6 unseen clients)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### The Three Leakage Traps & Empirical Attack Test

Following the leakage taxonomy in `skills/hunting-leakage-and-validating/SKILL.md`:
1. **Label-Derived Features:** Features computed from or mathematically coupled with the label. In our dataset, `trend_direction` is derived from `trend_pct`. Including `trend_pct` would allow the model to cheat effortlessly.
2. **Future / Overlapping Windows:** Features that aggregate activity during or after the prediction point. Using `impressions_last_30d` or `clicks_last_30d` leaks information from the outcome evaluation window.
3. **Decision-Derived Features:** Pre-existing product tags or manual heuristic scores.

#### The "Train-Without" Attack Test (The Confession)
To verify that our validation harness is genuinely sensitive, we execute the canonical attack test: we deliberately introduce the suspect leaky feature `trend_pct` into the training pipeline.
- If the harness is sound, the model will seize upon this shortcut: ROC-AUC and Precision@50 will skyrocket toward **1.000**, with `trend_pct` dominating feature importance.
- In our sanitized production model, `trend_pct` is quarantined. The score drops to **0.760**—confirming that our production model learns genuine behavioral signals rather than reading the answer key.

#### Concrete Failure Analysis (Real Model Errors)
Inspecting model errors on the unseen holdout partition reveals structural real-world limitations:
- **False Positives (High Predicted Decline Probability, Actual = Flat/Growing):**
  Legacy URLs with aged publication dates (`content_age_days > 150`) situated at striking distance (`avg_position ≈ 12-15`). The model flags them based on typical decay curves, but recent editorial refreshes (`days_since_last_update < 20`) allowed them to sustain momentum.
- **False Negatives (Low Predicted Decline Probability, Actual = Declining):**
  URLs with near-zero search visibility (`impressions_90d ≤ 3`, `avg_position = 0.0`). As documented in our data dictionary, `avg_position = 0` denotes missing rank data rather than rank zero. The model assigns low decline probability due to lack of search volume, missing subtle traffic drop-offs.


In [3]:
# Section 3 Code: Empirical Leakage Attack Test & Concrete Failure Analysis

# 1. Deliberately construct leaky feature set by injecting trend_pct
X_leaky = X_honest.copy()
X_leaky["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)

rf_leaky = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=25, random_state=42, n_jobs=-1
)
rf_leaky.fit(X_leaky[train_mask], y[train_mask])
probs_leaky = rf_leaky.predict_proba(X_leaky[test_mask])[:, 1]
res_leaky = evaluate_ranking(y[test_mask], probs_leaky)

print("=== The Leakage Attack Test (The Confession) ===")
attack_comparison = pd.DataFrame([res_honest, res_leaky], index=[
    "Honest Model (Strictly Prior Signals)",
    "Leaky Model (Injected trend_pct)"
])
print(attack_comparison.to_string())

# Feature importance confession
importances = pd.Series(rf_leaky.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)
print(f"\nTop Leaky Feature Importance: trend_pct accounts for {importances['trend_pct']:.2%} of tree splits!")
print("Top 5 features in leaky model:")
for feat, imp in importances.head(5).items():
    print(f"  - {feat:<25}: {imp:.4f}")

# 2. Extract Concrete Real-World Failure Examples from Holdout Test Set
test_df = df[test_mask].copy()
test_df["pred_prob"] = probs_honest
test_df["actual_label"] = y[test_mask]

# False Positives: Model predicted high decline probability (p > 0.7), but page did not decline (actual = 0)
fps = test_df[test_df["actual_label"] == 0].sort_values("pred_prob", ascending=False).head(3)

# False Negatives: Model predicted low decline probability (p < 0.2), but page declined (actual = 1)
fns = test_df[test_df["actual_label"] == 1].sort_values("pred_prob", ascending=True).head(3)

print("\n=== Real Failure Example 1: High-Confidence False Positives (Type I) ===")
print("Model predicted high decay risk, but content maintained traffic or grew:")
for _, r in fps.iterrows():
    print(f"  * ID: {r['content_id']} | Client: {r['client_id']} | Pred Prob: {r['pred_prob']:.3f} | Actual: {r['actual_label']} "
          f"| Age: {r['content_age_days']}d | Stale: {r['days_since_last_update']}d | Pos: {r['avg_position']} | Imp90d: {r['impressions_90d']:,}")

print("\n=== Real Failure Example 2: Low-Confidence False Negatives (Type II) ===")
print("Model predicted low decay risk, but content declined:")
for _, r in fns.iterrows():
    print(f"  * ID: {r['content_id']} | Client: {r['client_id']} | Pred Prob: {r['pred_prob']:.3f} | Actual: {r['actual_label']} "
          f"| Age: {r['content_age_days']}d | Stale: {r['days_since_last_update']}d | Pos: {r['avg_position']} | Imp90d: {r['impressions_90d']:,}")


=== The Leakage Attack Test (The Confession) ===
                                       Base Rate  ROC-AUC  PR-AUC  Precision@10  Precision@20  Precision@50
Honest Model (Strictly Prior Signals)      0.391    0.754   0.626           0.7          0.75          0.76
Leaky Model (Injected trend_pct)           0.391    1.000   1.000           1.0          1.00          1.00

Top Leaky Feature Importance: trend_pct accounts for 83.65% of tree splits!
Top 5 features in leaky model:
  - trend_pct                : 0.8365
  - days_with_impressions    : 0.0211
  - impressions_90d          : 0.0183
  - avg_position             : 0.0176
  - log_impressions_90d      : 0.0171

=== Real Failure Example 1: High-Confidence False Positives (Type I) ===
Model predicted high decay risk, but content maintained traffic or grew:
  * ID: content_8fdbff16a886 | Client: client_f74efabef1 | Pred Prob: 0.785 | Actual: 0 | Age: 175d | Stale: 20d | Pos: 12.0 | Imp90d: 3,863
  * ID: content_db1cd41b4b4f | Client: cl

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

When presenting the findings of this research, it is important to use language that is transparent and grounded in observed data, rather than making overly strong claims such as “guaranteeing” or “proving” an outcome. The model we developed is not intended to serve as an absolute predictor of Google’s algorithms, but rather as an operational decision-support system, which demonstrated a measured *Precision@50* of 0.760 when evaluated on a new client.

Furthermore, while we observed that content left unchanged for more than 180 days tends to experience a decline in visibility, this finding should be interpreted as a directional correlation rather than definitive evidence of causation. This is particularly important given that many evergreen pages continue to perform well without frequent updates. Ultimately, all performance metrics reported in this study are based on objective and measurable evaluations, rather than unsubstantiated promises of traffic recovery without experimental validation.

In [4]:
# Section 4 Code: Summary Verification Receipts for All Empirical Claims
print("=== Summary Verification Receipts (Audit Trail) ===")
print(f"1. Evaluated Cohort:               30,000 content items across 32 clients")
print(f"2. Holdout Partition Size:         {len(y[test_mask]):,} pages across {len(test_clients)} unseen clients")
print(f"3. Unseen Holdout Base Rate:       {res_honest['Base Rate']:.3f} ({res_honest['Base Rate']:.1%} random guess probability)")
print(f"4. Week-4 Heuristic Baseline:      Precision@50 = 0.440")
print(f"5. Honest Random Forest Model:     Precision@50 = {res_honest['Precision@50']:.3f} | ROC-AUC = {res_honest['ROC-AUC']:.3f}")
print(f"6. Measured Lift over Baseline:    +{res_honest['Precision@50'] - 0.440:.3f} ({res_honest['Precision@50'] - 0.440:+.1%} percentage points)")
print(f"7. Naive Random Split Score:       Precision@50 = {res_naive['Precision@50']:.3f} (Inflated by domain memorization)")
print(f"8. Measured Memorization Gap:      +{res_naive['Precision@50'] - res_honest['Precision@50']:.3f} percentage points")
print(f"9. Leakage Attack Confession:      Score leaps from {res_honest['Precision@50']:.3f} -> {res_leaky['Precision@50']:.3f} when trend_pct is added")
print(f"10. Target & Leakage Quarantine:   trend_direction, trend_pct, and *_last_30d strictly excluded from features")


=== Summary Verification Receipts (Audit Trail) ===
1. Evaluated Cohort:               30,000 content items across 32 clients
2. Holdout Partition Size:         2,325 pages across 6 unseen clients
3. Unseen Holdout Base Rate:       0.391 (39.1% random guess probability)
4. Week-4 Heuristic Baseline:      Precision@50 = 0.440
5. Honest Random Forest Model:     Precision@50 = 0.760 | ROC-AUC = 0.754
6. Measured Lift over Baseline:    +0.320 (+32.0% percentage points)
7. Naive Random Split Score:       Precision@50 = 0.940 (Inflated by domain memorization)
8. Measured Memorization Gap:      +0.180 percentage points
9. Leakage Attack Confession:      Score leaps from 0.760 -> 1.000 when trend_pct is added
10. Target & Leakage Quarantine:   trend_direction, trend_pct, and *_last_30d strictly excluded from features


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
